# Setup storage and shared tables

This notebook creates the shared storage and table setup for the workspace — everything that's identical regardless of which dataset gets onboarded, so it's created once here rather than duplicated in every dataset's setup notebook.

- Create the project catalog and shared bronze/silver/gold schemas.
- Create the shared landing and files volumes.
- Create the shared `bronze.ingestion_runs` table.
- Create the shared Silver tables (`silver.image_inventory`, `silver.leakage_groups`, `silver.rejected_records`) and their shared label columns.
- Create the shared `gold.manifest_rows` table and the `gold.manifest_registry` view over it (one row per manifest release, for a quick "what manifests do I have" overview).
- Leave dataset-specific paths and DDL (Bronze source metadata, any dataset-specific label axis) to the dataset setup notebooks.


In [ ]:
%run ./_setup_env


In [ ]:
from data_platform.labels import SILVER_LABEL_COLUMN_DDL
from data_platform.dataset_layout import load_yaml_config

STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
LANDING_ROOT = STORAGE_CONFIG["landing_root"]
STORAGE_ROOT = STORAGE_CONFIG["storage_root"]

# CATALOG_NAME is already set by _setup_env.ipynb (cell above)
MEDALLION_LAYERS = ["bronze", "silver", "gold"]


In [ ]:
# Catalog itself is created here (the one place that does). _setup_env.ipynb already tried
# USE CATALOG for every notebook including this one, but tolerates it not existing yet on a
# fresh workspace -- so re-select it here too, now that it definitely exists.
#
# Checked for existence explicitly rather than relying on `CREATE CATALOG IF NOT EXISTS`:
# Unity Catalog resolves the metastore's default storage root as an eager precondition of
# CREATE CATALOG, before it checks whether the catalog already exists -- so IF NOT EXISTS
# does not shield a re-run from a broken/missing default storage root once the catalog is
# already there. Skipping the call entirely when the catalog exists avoids that check.
existing_catalogs = {row.catalog for row in spark.sql("SHOW CATALOGS").collect()}
if CATALOG_NAME not in existing_catalogs:
    spark.sql(f"CREATE CATALOG {CATALOG_NAME}")
spark.sql(f"USE CATALOG {CATALOG_NAME}")

for layer in MEDALLION_LAYERS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{layer}")

spark.sql("CREATE VOLUME IF NOT EXISTS bronze.landing")
spark.sql("CREATE VOLUME IF NOT EXISTS bronze.files")

dbutils.fs.mkdirs(LANDING_ROOT)
dbutils.fs.mkdirs(f"{LANDING_ROOT.rstrip('/')}/archives")

display(spark.sql(f"SHOW VOLUMES IN {CATALOG_NAME}.bronze"))
display(spark.sql("SHOW SCHEMAS"))
display(dbutils.fs.ls(LANDING_ROOT))
display(dbutils.fs.ls(STORAGE_ROOT))

## Shared Bronze table

`bronze.ingestion_runs` — the one Bronze table that's shared across every dataset rather than dataset-prefixed (see `docs/data_contract.md`). Dataset-specific Bronze tables (`bronze.<dataset>_source_metadata`, `bronze.<dataset>_image_index`) are created in that dataset's own `05_setup_tables_and_folders.ipynb`, not here.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.ingestion_runs (
      ingestion_run_id STRING,
      dataset_name STRING,
      source_version STRING,
      started_at TIMESTAMP,
      finished_at TIMESTAMP,
      status STRING,
      records_seen INT,
      records_written INT
    )
    USING DELTA
    """
)

display(spark.sql("SHOW TABLES IN bronze"))


## Shared Silver tables

`silver.image_inventory`, `silver.leakage_groups`, `silver.rejected_records` — shared across every dataset, not dataset-prefixed (see `docs/decisions/002-silver-label-columns-not-map.md`). `silver.image_inventory` is created with only the universal columns first, then the shared label columns (`malignancy`, `specific_diagnosis` — `data_platform.labels.SILVER_LABEL_COLUMN_DDL`) are added via a guarded `ALTER TABLE ADD COLUMNS` (Delta has no `ADD COLUMNS IF NOT EXISTS`, so the guard is done in Python). A future dataset needing a genuinely different label axis defines its own DDL fragment and runs its own guarded `ALTER TABLE` from its own setup notebook instead of adding to `SILVER_LABEL_COLUMN_DDL`.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.image_inventory (
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      source_checksum STRING,
      image_width INT,
      image_height INT,
      image_format STRING,
      validation_status STRING,
      validation_reason STRING,
      patient_id STRING,
      lesion_id STRING,
      group_id STRING,
      validated_at TIMESTAMP
    )
    USING DELTA
    """
)

existing_columns = {field.name for field in spark.table("silver.image_inventory").schema.fields}
missing_label_columns = [
    f"{name} {ddl}" for name, ddl in SILVER_LABEL_COLUMN_DDL.items() if name not in existing_columns
]
if missing_label_columns:
    spark.sql(f"ALTER TABLE silver.image_inventory ADD COLUMNS ({', '.join(missing_label_columns)})")

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.leakage_groups (
      dataset_key STRING,
      group_id STRING,
      group_type STRING,
      group_source STRING,
      image_count INT
    )
    USING DELTA
    """
)
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.rejected_records (
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      rejection_reason STRING,
      rejected_at TIMESTAMP
    )
    USING DELTA
    """
)

display(spark.sql("SHOW TABLES IN silver"))
display(spark.sql("DESCRIBE TABLE silver.image_inventory"))


## Shared Gold tables

`gold.manifest_rows` — row-level detail, one row per image per manifest release. Shared across every dataset, versioned by `dataset_version` rather than dataset-prefixed (more than one release, e.g. `sample-v1` alongside a later full-scale `v1`, can coexist in this table — see `docs/data_contract.md`).

`gold.manifest_registry` — a view over `gold.manifest_rows`, one row per `dataset_version`, summarizing which datasets/labels/splits it contains. This is a view, not a table: everything in it is fully derivable from `manifest_rows`, so it can never drift out of sync and needs no separate write path. Query it to answer "which manifests do I have, and what's in each one" without hand-writing the aggregation each time.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS gold.manifest_rows (
      dataset_version STRING,
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      source_checksum STRING,
      label STRING,
      group_id STRING,
      split STRING,
      sample_seed INT,
      split_seed INT,
      preprocessing_version STRING,
      manifest_row_hash STRING,
      created_at TIMESTAMP,
      git_commit STRING
    )
    USING DELTA
    """
)
# Tables created before git_commit was added get it here (Delta has no ADD COLUMNS IF NOT EXISTS).
if "git_commit" not in {field.name for field in spark.table("gold.manifest_rows").schema.fields}:
    spark.sql("ALTER TABLE gold.manifest_rows ADD COLUMNS (git_commit STRING)")

spark.sql(
    """
    CREATE OR REPLACE VIEW gold.manifest_registry AS
    SELECT
      dataset_version,
      sort_array(collect_set(dataset_key)) AS dataset_keys,
      COUNT(*) AS image_count,
      SUM(CASE WHEN split = 'train' THEN 1 ELSE 0 END) AS train_count,
      SUM(CASE WHEN split = 'validation' THEN 1 ELSE 0 END) AS validation_count,
      SUM(CASE WHEN split = 'test' THEN 1 ELSE 0 END) AS test_count,
      sort_array(collect_set(label)) AS label_values,
      sort_array(collect_set(preprocessing_version)) AS preprocessing_versions,
      sort_array(collect_set(sample_seed)) AS sample_seeds,
      sort_array(collect_set(split_seed)) AS split_seeds,
      MIN(created_at) AS first_created_at,
      MAX(created_at) AS last_updated_at,
      sort_array(collect_set(git_commit)) AS git_commits
    FROM gold.manifest_rows
    GROUP BY dataset_version
    """
)

display(spark.sql("SHOW TABLES IN gold"))
display(spark.sql("DESCRIBE TABLE gold.manifest_rows"))
display(spark.sql("SELECT * FROM gold.manifest_registry"))


## Storage conventions

- Landing volume convention: `archives/<source-archive>.zip`
- Bronze storage convention: `<dataset_key>/...`
- Gold storage convention: `gold/<dataset_version>/shards/<split>/...` — a derived, fully rebuildable MosaicML shard export (`notebooks/31_export_gold_shards.ipynb`); see `docs/decisions/004-stream-archives-no-blob-storage.md`. How long an export is kept around is undecided — see `docs/decisions/006-gold-shard-retention-undecided.md`


## Dataset-specific setup

Dataset notebooks should create dataset-specific paths and dataset-specific tables.

Current example:

- `notebooks/isic_2019/05_setup_tables_and_folders.ipynb`
- `notebooks/isic_2019/10_bronze_ingest.ipynb`

That notebook should load dataset config, create dataset-specific DDL, and manage dataset-specific folder paths.